# Reduced-Order Modeling of Hypersonic Flat-Plate Flow Using POD and Machine Learning Surrogates

This notebook implements a non-intrusive Reduced-Order Model (ROM) using Proper Orthogonal Decomposition (POD) and three different machine learning surrogates (RBF, GPR, and NN) to predict surface distributions of Stanton number ($St$) and Skin Friction coefficient ($C_f$) in hypersonic rarefied Argon flow.

In [ ]:
# @title Setup and Library Installation
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import Rbf
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, WhiteKernel
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
import os

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

## 1. Dataset Acquisition

In [ ]:
# @title Dataset Acquisition
import os

# Google Drive folder ID from the provided link
FOLDER_ID = '1eNte1-tGO59HKiM6hCnQr8RothDeGmCF'

if not os.path.exists('Data'):
    print("Downloading dataset from Google Drive...")
    # gdown is pre-installed in Colab; --folder downloads the entire directory
    # We output it directly to a folder named 'Data'
    !gdown --folder {FOLDER_ID} -O Data
else:
    print("Dataset already exists in the environment.")

def get_case_names(path):
    with open(path, 'r') as f:
        header = f.readline()
    return header.strip("# \n").split()

DATA_DIR = "Data"
ST_DIR = os.path.join(DATA_DIR, "St")
CF_DIR = os.path.join(DATA_DIR, "Cf")
WALL_X_PATH = os.path.join(DATA_DIR, "wall_x.txt")

## 2. Data Loading and Preprocessing

In [ ]:
# @title Data Loading and Preprocessing
print("Loading data...")

x_coords = np.loadtxt(WALL_X_PATH)
x_mm = x_coords * 1000
time_indices = range(3, 11)
n_h = len(x_coords)

def build_snapshot_matrix(folder, prefix):
    snapshots = []
    case_list = []
    for t in time_indices:
        fpath = os.path.join(folder, f"{prefix}_t{t}.dat")
        data = np.loadtxt(fpath)
        if not case_list:
            case_list = get_case_names(fpath)
        snapshots.append(data)
    return np.hstack(snapshots), case_list

S_st, case_names = build_snapshot_matrix(ST_DIR, "stanton")
S_cf, _ = build_snapshot_matrix(CF_DIR, "cf")

n_cases = len(case_names)
n_snaps = len(time_indices)

val_cases = ['case4', 'case16']
test_cases = ['test1', 'valcase']
train_cases = [c for c in case_names if c not in val_cases and c not in test_cases]

def get_indices(names):
    idxs = []
    for name in names:
        base_idx = case_names.index(name)
        case_snap_idxs = [base_idx + i*n_cases for i in range(n_snaps)]
        idxs.extend(case_snap_idxs)
    return sorted(idxs)

train_idxs = get_indices(train_cases)
val_idxs = get_indices(val_cases)
test_idxs = get_indices(test_cases)

## 3. Proper Orthogonal Decomposition (POD)

In [ ]:
# @title Proper Orthogonal Decomposition (POD)
def perform_pod(S_full, train_idxs, energy_threshold=0.999):
    S_train = S_full[:, train_idxs]
    mean_vec = np.mean(S_train, axis=1, keepdims=True)
    S_prime = S_train - mean_vec
    U, Sigma, VT = np.linalg.svd(S_prime, full_matrices=False)
    cumulative_energy = np.cumsum(Sigma**2) / np.sum(Sigma**2)
    L = np.argmax(cumulative_energy >= energy_threshold) + 1
    Phi = U[:, :L]
    Alpha = Phi.T @ (S_full - mean_vec)
    return Phi, mean_vec, Alpha, L

Phi_st, mean_st, Alpha_st, L_st = perform_pod(S_st, train_idxs)
Phi_cf, mean_cf, Alpha_cf, L_cf = perform_pod(S_cf, train_idxs)
print(f"POD Modes kept for St: {L_st}, Cf: {L_cf}")

## 4. Feature Engineering

In [ ]:
# @title Feature Engineering
case_physics = {
    'case1': [0.012, 9.60], 'case2': [0.018, 8.81], 'case3': [0.025, 8.49],
    'case4': [0.035, 7.97], 'case5': [0.045, 7.75], 'test1': [0.050, 7.61],
    'case6': [0.055, 7.52], 'case7': [0.065, 7.69], 'case8': [0.075, 7.89],
    'case9': [0.085, 8.12], 'case10': [0.095, 8.54], 'case11': [0.120, 9.00],
    'test2': [0.150, 8.51], 'case12': [0.180, 8.34], 'case13': [0.250, 8.75],
    'case14': [0.350, 8.66], 'case15': [0.450, 8.54], 'case16': [0.550, 8.58],
    'case17': [0.650, 8.69], 'valcase': [0.005, 12.65]
}
time_vals = [0.00015, 0.0002, 0.00025, 0.0003, 0.00035, 0.0004, 0.00045, 0.0005]

X_features = []
for t_val in time_vals:
    for cname in case_names:
        kn, ma = case_physics[cname]
        tau = t_val * 1600 / 0.055
        X_features.append([np.log10(kn), ma, tau])

X_features = np.array(X_features)
Y_joint = np.vstack([Alpha_st, Alpha_cf]).T

scaler_x = StandardScaler()
scaler_y = StandardScaler()
X_train = scaler_x.fit_transform(X_features[train_idxs])
Y_train = scaler_y.fit_transform(Y_joint[train_idxs])
X_val = scaler_x.transform(X_features[val_idxs])
Y_val = scaler_y.transform(Y_joint[val_idxs])
X_test = scaler_x.transform(X_features[test_idxs])
Y_test = scaler_y.transform(Y_joint[test_idxs])

## 5. Surrogate Training

In [ ]:
# @title Surrogate Training
print("Training RBF...")
rbf_model = Rbf(X_train[:,0], X_train[:,1], X_train[:,2], Y_train, function='thin_plate', mode='N-D')

print("Training GPR...")
kernel = 1.0 * Matern(length_scale=1.0, nu=2.5) + WhiteKernel(noise_level=1e-5)
gpr = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=10)
gpr.fit(X_train, Y_train)

print("Training NN...")
nn = models.Sequential([
    layers.Input(shape=(3,)),
    layers.Dense(64, activation='elu'),
    layers.BatchNormalization(),
    layers.Dense(64, activation='elu'),
    layers.BatchNormalization(),
    layers.Dense(32, activation='elu'),
    layers.Dense(Y_train.shape[1])
])
nn.compile(optimizer='adam', loss='mse')
early_stop = callbacks.EarlyStopping(patience=100, restore_best_weights=True)
nn.fit(X_train, Y_train, validation_data=(X_val, Y_val), epochs=1000, verbose=0, callbacks=[early_stop])

## 6. Model Evaluation and Comparison

In [ ]:
# @title Model Evaluation and Comparison
def reconstruct(y_pred_scaled, Phi_st, mean_st, L_st, Phi_cf, mean_cf):
    y_pred = scaler_y.inverse_transform(y_pred_scaled)
    alpha_st_p = y_pred[:, :L_st].T
    alpha_cf_p = y_pred[:, L_st:].T
    rec_st = Phi_st @ alpha_st_p + mean_st
    rec_cf = Phi_cf @ alpha_cf_p + mean_cf
    return rec_st, rec_cf

y_rbf = rbf_model(X_test[:,0], X_test[:,1], X_test[:,2])
y_gpr, _ = gpr.predict(X_test, return_std=True)
y_nn = nn.predict(X_test)

idx_in_test = -1
st_rbf, cf_rbf = reconstruct(y_rbf, Phi_st, mean_st, L_st, Phi_cf, mean_cf)
st_gpr, cf_gpr = reconstruct(y_gpr, Phi_st, mean_st, L_st, Phi_cf, mean_cf)
st_nn, cf_nn = reconstruct(y_nn, Phi_st, mean_st, L_st, Phi_cf, mean_cf)

fig, ax = plt.subplots(1, 2, figsize=(15, 5))
ax[0].plot(x_mm, S_st[:, test_idxs[idx_in_test]], 'r-', label='High-Fidelity')
ax[0].plot(x_mm, st_rbf[:, idx_in_test], 'g--', label='POD-RBF')
ax[0].plot(x_mm, st_gpr[:, idx_in_test], 'b:', label='POD-GPR')
ax[0].plot(x_mm, st_nn[:, idx_in_test], 'k-.', label='POD-NN')
ax[0].set_title("Stanton Number: Valcase (Extrapolation)")
ax[0].set_xlabel("x [mm]")
ax[0].legend()

ax[1].plot(x_mm, S_cf[:, test_idxs[idx_in_test]], 'r-', label='High-Fidelity')
ax[1].plot(x_mm, cf_rbf[:, idx_in_test], 'g--', label='POD-RBF')
ax[1].plot(x_mm, cf_gpr[:, idx_in_test], 'b:', label='POD-GPR')
ax[1].plot(x_mm, cf_nn[:, idx_in_test], 'k-.', label='POD-NN')
ax[1].set_title("Skin Friction: Valcase (Extrapolation)")
ax[1].set_xlabel("x [mm]")
ax[1].legend()
plt.tight_layout()
plt.show()